### Cell 1: load the data

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

data = pd.read_csv("../data/processed/model_table.csv")
print(data.shape)
data.head()

(45, 10)


,Country Code,country,IncomeGroup,life_expectancy,health_spending_pct_gdp,electricity_access,basic_water,basic_sanitation,urban_population_pct,log_gdp_per_capita
0,AGO,Angola,Lower middle income,63.051,2.203943,45.6,64.159911,49.547024,66.663782,7.827188
1,BDI,Burundi,Low income,62.180,9.059406,9.8,63.757991,44.357739,18.786178,5.456648
2,BEN,Benin,Lower middle income,59.888,3.002069,40.0,67.983252,18.842084,48.712550,7.030700
3,BFA,Burkina Faso,Low income,60.184,5.350011,17.6,50.699617,26.718569,26.118725,6.639684
4,BWA,Botswana,Upper middle income,67.167,6.121932,70.1,91.474948,69.815060,66.327169,8.877952


### Cell 2: the simplest model, one predictor

In [3]:
y = data["life_expectancy"]
X = sm.add_constant(data[["log_gdp_per_capita"]])

simple = sm.OLS(y, X).fit()
print(simple.summary())

                            OLS Regression Results                            
Dep. Variable:        life_expectancy   R-squared:                       0.223
Model:                            OLS   Adj. R-squared:                  0.205
Method:                 Least Squares   F-statistic:                     12.32
Date:                Wed, 28 Oct 2026   Prob (F-statistic):            0.00107
Time:                        14:04:10   Log-Likelihood:                -143.16
No. Observations:                  45   AIC:                             290.3
Df Residuals:                      43   BIC:                             293.9
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                 38.3753      6

### Cell 3: the multiple regression with all six predictors

In [4]:
features = ["log_gdp_per_capita", "health_spending_pct_gdp", "electricity_access",
            "basic_water", "basic_sanitation", "urban_population_pct"]

X = sm.add_constant(data[features])
multi = sm.OLS(y, X).fit()
print(multi.summary())

                            OLS Regression Results                            
Dep. Variable:        life_expectancy   R-squared:                       0.459
Model:                            OLS   Adj. R-squared:                  0.373
Method:                 Least Squares   F-statistic:                     5.369
Date:                Wed, 28 Oct 2026   Prob (F-statistic):           0.000427
Time:                        14:04:58   Log-Likelihood:                -135.02
No. Observations:                  45   AIC:                             284.0
Df Residuals:                      38   BIC:                             296.7
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const                     

### Cell 4: a side-by-side comparison

In [5]:
comparison = pd.DataFrame({
    "coefficient": multi.params,
    "p_value": multi.pvalues,
})
comparison.round(3)

,coefficient,p_value
const,57.044,0.000
log_gdp_per_capita,-0.862,0.596
health_spending_pct_gdp,-0.569,0.044
electricity_access,0.058,0.375
basic_water,0.161,0.134
basic_sanitation,0.095,0.126
urban_population_pct,-0.071,0.257


### Cell 5: VIF, which measures predictor overlap

In [6]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X = sm.add_constant(data[features])
vif = pd.DataFrame({
    "feature": X.columns,
    "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])],
})
vif[vif["feature"] != "const"].round(2)

,feature,VIF
1,log_gdp_per_capita,3.66
2,health_spending_pct_gdp,1.19
3,electricity_access,4.16
4,basic_water,3.88
5,basic_sanitation,2.71
6,urban_population_pct,1.65


### Cell 6: which countries have the most influence?

In [7]:
data["cooks_d"] = multi.get_influence().cooks_distance[0]
print("Threshold (4/n):", round(4 / len(data), 3))
data.sort_values("cooks_d", ascending=False)[["country", "life_expectancy", "cooks_d"]].head(5)

Threshold (4/n): 0.089


,country,life_expectancy,cooks_d
5,Central African Republic,31.530,0.727974
12,Ethiopia,65.741,0.072022
22,Mali,59.217,0.055799
29,Nigeria,53.009,0.050832
36,Eswatini,59.850,0.050709


### Cell 7: refit with and without the outliers

In [8]:
def fit(df, cols):
    X = sm.add_constant(df[cols])
    return sm.OLS(df["life_expectancy"], X).fit()

no_caf = data[data["Country Code"] != "CAF"]
no_caf_lso = no_caf[no_caf["Country Code"] != "LSO"]

models = {"all_45": data, "no_CAF": no_caf, "no_CAF_LSO": no_caf_lso}

print("Simple model (log GDP only):")
for name, df in models.items():
    m = fit(df, ["log_gdp_per_capita"])
    print(f"  {name}: slope={m.params['log_gdp_per_capita']:.2f}, R2={m.rsquared:.3f}, n={len(df)}")

print("\nSix-predictor model:")
table = pd.DataFrame({name: fit(df, features).params for name, df in models.items()})
r2 = {name: round(fit(df, features).rsquared_adj, 3) for name, df in models.items()}
print("Adjusted R2:", r2)
table.round(3)

Simple model (log GDP only):
  all_45: slope=3.33, R2=0.223, n=45
  no_CAF: slope=2.49, R2=0.240, n=44
  no_CAF_LSO: slope=2.43, R2=0.245, n=43

Six-predictor model:
Adjusted R2: {'all_45': np.float64(0.373), 'no_CAF': np.float64(0.407), 'no_CAF_LSO': np.float64(0.4)}


,all_45,no_CAF,no_CAF_LSO
const,57.044,61.692,60.666
log_gdp_per_capita,-0.862,-0.540,-0.577
health_spending_pct_gdp,-0.569,-0.292,0.132
electricity_access,0.058,0.062,0.078
basic_water,0.161,-0.010,-0.035
basic_sanitation,0.095,0.119,0.110
urban_population_pct,-0.071,-0.006,0.009
